# `DocLayNet` EDA, and `torchvision` transformations
**Author:** Juan Pablo Triana Martinez
**Date:** 2026-03-18

After a successful implementation of `src.utils.data_utils` in order to obtain sub-sampled datasets, small enough to perform proper experimentations, but also big enough to do deep learning tasks in google collabs, the next step would be to perform an **exploratory data analysis** to be able to do the following:
* Read the `train.json`, `val.json`, and `test.json` files from a specified `COCO` folder, from specific subset-dataset.
* from the `supercategory` key, inside any of the `.json` files, retrieve the following:
    - `id` as `category_id`
    - `name` as `category_name`
* from the `images` key, inside any of the `.json` files, retrieve the following:
    - `id` as `image_id`
    - `file_name` as `image_name`
    - `doc_name` as `doc_name`
    - `doc_category` as `doc_category`
* from the `annotations` key, inside any of the `.json` foles., retrieve the following:
    - `bbox` as `bbox_coords`
    - `segmentation` as `seg_coords`
* from each of the `.json` unique files inside `JSON` folder, from a specific subset-dataset.
    - from `cells` key, retrieve `bbox` as `cell_bboxes` and `text` as `cell_texts`

With this in mind, we are gonna read all of the images and do this:
1. Plot the original image in `PIL` format, with respective annotations.
2. Convert said image to `torch.tensor`. The original size is `(1025 by 1025)`
3. Apply `torchvision` resizing to the followingL
    - Shape of `(3, 1024, 1024)`
    - Shape of `(3, 512, 512)`
    - Shape of `(3, 256, 256)`
    - Shape of `(3, 224, 224)`

**NOTE:** Its important to also scale down the `bbox_coords` and `seg_coords`. Why? because scaling by these factors will indeed create a change where the areas of text are!

<img src="../docs/chatgpt_bbox_coords.png" width="550">
<img src="../docs/chatgpt_bbox_example.png" width="500">

```python

```

## 1. Read the `train.json`, `test.json`, and `val.json` files inside a `COCO` folder
Now that we have setup the specific dataset, we are gonna retrieve the `.json` metadata

In [2]:
from pathlib import Path
import json

In [ ]:
# Let's get the data folder path
data_folder_path = Path().cwd().parent / "data"

# Let's retrieve a specific filename
filename = "test_subsample_seed_42"
subset_path = data_folder_path / filename

True

In [43]:
from typing import Union, List, Tuple
class MetadataRetriever:
    '''
    Class that will retrieve the most relevant metadata
    features from the train, val, and test.json files coming
    inside the COCO folder of a sub-dataset.
    '''
    def __init__(self,
                 data_path: Path,
                 dataset_name:str,
                 split_analyze:str = "train"):
        '''
        Args:
            - data_path (Path): Pathlib path towards the data folder path
            - dataset_name (str): string name of the dataset name.
        '''
        self.data_path = data_path
        self.dataset_name = dataset_name

        # Let's get the COCO folder
        self.coco_folder_path = self.data_path / self.dataset_name / "COCO"
        assert self.coco_folder_path.exists(), f"the COCO folder inside the dataset named: {self.dataset_name} doesn't exist inside {self.data_path}, please take a look."

        assert split_analyze in ["train", "val", "test"], "Please type a valud split, either train, val, or test"
        self.split = split_analyze

        if split_analyze == "train":
            self.json_to_analyze = "train.json"
        if split_analyze == "test":
            self.json_to_analyze = "test.json"
        if split_analyze == "val":
            self.json_to_analyze = "val.json"
    
    def get_metadata_supercategories(self) -> List[dict[str, Union[int, str]]]:
        '''
        Instance method that will retrieve the available supercategories
        inside the .json file, in the COCO folder. 
        '''
        # Let's get the coco_path and categories key
        json_path = self.coco_folder_path / self.json_to_analyze
        with open(json_path, "r") as f:
            coco_dict = json.load(f)
        return coco_dict["categories"]

    def get_metadata_images(self) -> List[dict[str, Union[int, str]]]:
        '''
        Instance method that will retrieve the available images metadata
        inside the .json file, in the COCO folder. 
        '''
        # Let's get the coco_path and images key
        json_path = self.coco_folder_path / self.json_to_analyze
        with open(json_path, "r") as f:
            coco_dict = json.load(f)
        return coco_dict["images"]

    def get_metadata_annotations(self) -> List[dict[str, Union[int, str]]]:
        '''
        Instance method that will retrieve the available annotations metadata
        inside the .json file, in the COCO folder. 
        '''
        # Let's get the coco_path and annotations key
        json_path = self.coco_folder_path / self.json_to_analyze
        with open(json_path, "r") as f:
            coco_dict = json.load(f)
        return coco_dict["annotations"]

    def get_metadata_spec_image_id(self, img_id:int) ->Tuple[List[dict[str, Union[int, str]]], List[dict[str, Union[int, str]]]]:
        '''
        Instance method that will be used to retrieve the metadata
        from the core folders, these are inside COCO folder.
        Arg:
            - img_id (int): Integer that will be used to retrieve the specific image
        '''

        # Retrieve the images_dict and annotations
        images_dict = self.get_metadata_images()
        annotations_dict = self.get_metadata_annotations()

        # Let's check wether the image_id exists, else assert that please add a valid one
        available_image_ids = [x["id"] for x in images_dict]
        assert img_id in available_image_ids, f"Passed image id -> {img_id} does not exist inside {self.json_to_analyze}, inside {self.dataset_name} dataset. double check"
        
        # Select the image dictionary correspondent to the selected image id
        sel_image_dict = [img_dict for img_dict in images_dict if img_dict["id"] == img_id]

        # Select the annotations correspondent to the selected image id
        sel_annotations_dict = [annotation for annotation in annotations_dict if annotation["image_id"] == img_id]
        
        return sel_image_dict, sel_annotations_dict

    def get_metadata_extra_image_id(self, img_id:int) -> Tuple[List[dict[str, Union[int, str]]], List[dict[str, Union[int, str]]]]:
        '''
        Instance method that will retrieve the metadata from the extra
        folders, these are inside JSON folder.
        '''
        extra_json_folder_path = self.data_path / self.dataset_name / "JSON"
        assert extra_json_folder_path.exists(), f"JSON folder doesnt exist inside {self.dataset_name}"

        # Retrieve the images_dict and annotations
        images_dict = self.get_metadata_images()  

        # Let's check wether the image_id exists, else assert that please add a valid one
        available_image_ids = [x["id"] for x in images_dict]
        assert img_id in available_image_ids, f"Passed image id -> {img_id} does not exist inside {self.json_to_analyze}, inside {self.dataset_name} dataset. double check"

        # Select the image dictionary correspondent to the selected image id
        sel_image_dict = [img_dict for img_dict in images_dict if img_dict["id"] == img_id]
        hash_name :str = sel_image_dict[0]["file_name"].split(".")[0]

        # extra_json_name to access inside JSON folder
        extra_json_name = hash_name + ".json"

        with open(extra_json_folder_path / extra_json_name, "r") as f:
            extra_dict = json.load(f)
        
        return extra_dict["metadata"], extra_dict["cells"]


### 1.1 Let's instantiate a metadata_extractor and `get_metadata_supercategories()`

In [44]:
metadata_extractor = MetadataRetriever(
    data_path=data_folder_path,
    dataset_name=filename,
    split_analyze="test")

In [45]:
supercategories_dict = metadata_extractor.get_metadata_supercategories()

In [46]:
supercategories_dict

[{'supercategory': 'Caption', 'id': 1, 'name': 'Caption'},
 {'supercategory': 'Footnote', 'id': 2, 'name': 'Footnote'},
 {'supercategory': 'Formula', 'id': 3, 'name': 'Formula'},
 {'supercategory': 'List-item', 'id': 4, 'name': 'List-item'},
 {'supercategory': 'Page-footer', 'id': 5, 'name': 'Page-footer'},
 {'supercategory': 'Page-header', 'id': 6, 'name': 'Page-header'},
 {'supercategory': 'Picture', 'id': 7, 'name': 'Picture'},
 {'supercategory': 'Section-header', 'id': 8, 'name': 'Section-header'},
 {'supercategory': 'Table', 'id': 9, 'name': 'Table'},
 {'supercategory': 'Text', 'id': 10, 'name': 'Text'},
 {'supercategory': 'Title', 'id': 11, 'name': 'Title'}]

### 1.2 `get_metadata_images()`

In [47]:
images_dict = metadata_extractor.get_metadata_images()

In [48]:
images_dict

[{'id': 1078,
  'width': 1025,
  'height': 1025,
  'file_name': '3094a763486b7128d42712c411b426b00042c49b0186ab6e3673bf04aae043e0.png',
  'collection': 'ann_reports_10_14_fancy',
  'doc_name': 'NYSE_SMFG_2011.pdf',
  'page_no': 159,
  'precedence': 0,
  'doc_category': 'financial_reports'},
 {'id': 232,
  'width': 1025,
  'height': 1025,
  'file_name': '6ab9e42c621b08ad07d25985707c9fe1eb56bb7dd2c0724868ea67b0158ba490.png',
  'collection': 'ann_reports_00_04_fancy',
  'doc_name': 'ASX_STO_2004.pdf',
  'page_no': 25,
  'precedence': 0,
  'doc_category': 'financial_reports'}]

### 1.3 `get_metadata_annotations()`

In [49]:
annotations_dict = metadata_extractor.get_metadata_annotations()
annotations_dict

[{'id': 2986,
  'image_id': 232,
  'category_id': 8,
  'bbox': [52.77036038562963,
   56.97416718444981,
   851.8757332522538,
   28.776438782818445],
  'segmentation': [[52.77036038562963,
    56.97416718444981,
    52.77036038562963,
    85.75060596726826,
    904.6460936378834,
    85.75060596726826,
    904.6460936378834,
    56.97416718444981]],
  'area': 24513.949888502055,
  'iscrowd': 0,
  'precedence': 0},
 {'id': 2987,
  'image_id': 232,
  'category_id': 10,
  'bbox': [52.77036038562963,
   168.77876992935,
   191.29246346744912,
   57.79072229974679],
  'segmentation': [[52.77036038562963,
    168.77876992935,
    52.77036038562963,
    226.5694922290968,
    244.06282385307875,
    226.5694922290968,
    244.06282385307875,
    168.77876992935]],
  'area': 11054.92963428181,
  'iscrowd': 0,
  'precedence': 0},
 {'id': 2988,
  'image_id': 232,
  'category_id': 10,
  'bbox': [52.77036038562963,
   239.77504836797164,
   206.77698411530318,
   89.34463697515343],
  'segmentati

### 1.4 `get_metadata_spec_image_id` for available id
we will try also not an available id if assert works

In [50]:
sel_id = 42
sel_image_dict, sel_annotations_dict = metadata_extractor.get_metadata_spec_image_id(img_id=sel_id)

AssertionError: Passed image id -> 42 does not exist inside test.json, inside test_subsample_seed_42 dataset. double check

In [51]:
sel_id = 1078
sel_image_dict, sel_annotations_dict = metadata_extractor.get_metadata_spec_image_id(img_id=sel_id)

In [52]:
sel_image_dict

[{'id': 1078,
  'width': 1025,
  'height': 1025,
  'file_name': '3094a763486b7128d42712c411b426b00042c49b0186ab6e3673bf04aae043e0.png',
  'collection': 'ann_reports_10_14_fancy',
  'doc_name': 'NYSE_SMFG_2011.pdf',
  'page_no': 159,
  'precedence': 0,
  'doc_category': 'financial_reports'}]

In [53]:
sel_annotations_dict

[{'id': 14109,
  'image_id': 1078,
  'category_id': 6,
  'bbox': [37.16831866169205,
   18.11225818176206,
   44.849817368630546,
   10.193164808827987],
  'segmentation': [[37.16831866169205,
    18.11225818176206,
    37.16831866169205,
    28.305422990590046,
    82.0181360303226,
    28.305422990590046,
    82.0181360303226,
    18.11225818176206]],
  'area': 457.1615800842871,
  'iscrowd': 0,
  'precedence': 0},
 {'id': 14110,
  'image_id': 1078,
  'category_id': 8,
  'bbox': [71.51163365654716,
   47.15607542583518,
   384.0553266912568,
   19.75876488449387],
  'segmentation': [[71.51163365654716,
    47.15607542583518,
    71.51163365654716,
    66.91484031032905,
    455.56696034780396,
    66.91484031032905,
    455.56696034780396,
    47.15607542583518]],
  'area': 7588.458902730027,
  'iscrowd': 0,
  'precedence': 0},
 {'id': 14111,
  'image_id': 1078,
  'category_id': 8,
  'bbox': [73.1934807847817,
   101.45603163478495,
   321.53218878194184,
   10.274511248877161],
  's

### 1.5 `get_metadata_extra_image_id`

In [54]:
extra_dict_metadata, extra_dict_cells = metadata_extractor.get_metadata_extra_image_id(img_id=sel_id)

In [55]:
extra_dict_metadata

{'page_hash': '3094a763486b7128d42712c411b426b00042c49b0186ab6e3673bf04aae043e0',
 'original_filename': 'NYSE_SMFG_2011.pdf',
 'page_no': 159,
 'num_pages': 230,
 'original_width': 609.44897,
 'original_height': 793.70099,
 'coco_width': 1025,
 'coco_height': 1025,
 'collection': 'ann_reports_10_14_fancy',
 'doc_category': 'financial_reports'}

In [56]:
extra_dict_cells

[{'bbox': [37.16831866169205,
   18.11225818176206,
   44.849817368630546,
   10.193164808827987],
  'text': 'SMBC',
  'font': {'color': [0, 0, 0, 255],
   'name': '/MCCALN+HelveticaNeueLTStd-Md',
   'size': 1}},
 {'bbox': [94.64678974681013,
   992.6928461434828,
   42.86692464998342,
   9.24655517942594],
  'text': 'SMFG ',
  'font': {'color': [0, 0, 0, 255],
   'name': '/MCCAKK+HelveticaNeueLTStd-Bd',
   'size': 1}},
 {'bbox': [137.51371439679355,
   993.0337805802661,
   30.89216132402356,
   9.29821320369022],
  'text': '2011',
  'font': {'color': [0, 0, 0, 255],
   'name': '/MCCAKL+Caslon224Std-Medium',
   'size': 1}},
 {'bbox': [43.29074508075713,
   991.0501607475632,
   42.52382615397644,
   16.59730826340534],
  'text': '158',
  'font': {'color': [128, 130, 133, 255],
   'name': '/MBPMMF+ITCAvantGardeStd-Demi',
   'size': 1}},
 {'bbox': [71.51163365654716,
   47.15607542583518,
   384.0553266912568,
   19.75876488449387],
  'text': 'Assets and Liabilities (Consolidated)',
  '